# SnapShelf — Train YOLO Models (Experiment 2)

This notebook trains two YOLO models on a free Colab GPU:
1. **14-class YOLO** (Pipeline B) — detects and classifies 14 fruit/veg classes
2. **Objectness YOLO** (Pipeline C) — detects objects without classifying them (1 class)

**Before running:** Create a `SnapShelf` folder in Google Drive and upload `dataset_14class.zip` and `dataset_objectness.zip` there.

**Results are saved to:** `Google Drive → SnapShelf/results/` (training logs, charts, weights)

## Step 1: Check GPU & Install Dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change runtime to GPU!'}")

In [ ]:
# Install ultralytics
!pip install ultralytics -q

## Step 2: Mount Google Drive & Extract Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

DRIVE = '/content/drive/MyDrive/SnapShelf'

# Extract 14-class dataset
print("Extracting 14-class dataset...")
with zipfile.ZipFile(f'{DRIVE}/dataset_14class.zip', 'r') as z:
    z.extractall('/content/')
print(f"  Train images: {len(os.listdir('/content/dataset/train/images'))}")
print(f"  Val images:   {len(os.listdir('/content/dataset/val/images'))}")

# Extract objectness dataset
print("\nExtracting objectness dataset...")
with zipfile.ZipFile(f'{DRIVE}/dataset_objectness.zip', 'r') as z:
    z.extractall('/content/')
print(f"  Train images: {len(os.listdir('/content/dataset_objectness/train/images'))}")
print(f"  Val images:   {len(os.listdir('/content/dataset_objectness/val/images'))}")

## Step 3: Create YOLO Data Configs

In [ ]:
# 14-class config
config_14 = """path: /content/dataset
train: train/images
val: val/images

nc: 14
names:
  0: apple
  1: banana
  2: bell_pepper_green
  3: bell_pepper_red
  4: carrot
  5: cucumber
  6: grape
  7: lemon
  8: onion
  9: orange
  10: peach
  11: potato
  12: strawberry
  13: tomato
"""

with open('/content/yolo_14class.yaml', 'w') as f:
    f.write(config_14)
print("Written: /content/yolo_14class.yaml")

# Objectness config
config_obj = """path: /content/dataset_objectness
train: train/images
val: val/images

nc: 1
names:
  0: object
"""

with open('/content/yolo_objectness.yaml', 'w') as f:
    f.write(config_obj)
print("Written: /content/yolo_objectness.yaml")

## Step 4: Train 14-Class YOLO (Pipeline B)

This takes ~1-2 hours on a T4 GPU.

In [ ]:
from ultralytics import YOLO

# Set seed for reproducibility
import random, numpy as np
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

model_14 = YOLO('yolov8s.pt')

results_14 = model_14.train(
    data='/content/yolo_14class.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    patience=15,
    lr0=0.01,
    project='/content/runs/yolo_14class',
    name='train',
    exist_ok=True,
    seed=42,
    verbose=True,
)

## Step 5: Train Objectness YOLO (Pipeline C)

This is faster since it's only 1 class.

In [ ]:
# Reset seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

model_obj = YOLO('yolov8s.pt')

results_obj = model_obj.train(
    data='/content/yolo_objectness.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    patience=15,
    lr0=0.01,
    project='/content/runs/yolo_objectness',
    name='train',
    exist_ok=True,
    seed=42,
    verbose=True,
)

## Step 6: Save Weights to Google Drive

Copy the best weights so you can download them.

In [ ]:
import shutil

DRIVE = '/content/drive/MyDrive/SnapShelf'

# Save weights
drive_weights = f'{DRIVE}/weights'
os.makedirs(drive_weights, exist_ok=True)

src_14 = '/content/runs/yolo_14class/train/weights/best.pt'
dst_14 = f'{drive_weights}/yolo_14class_best.pt'
shutil.copy2(src_14, dst_14)
print(f'Saved: {dst_14}  ({os.path.getsize(dst_14) / 1e6:.1f} MB)')

src_obj = '/content/runs/yolo_objectness/train/weights/best.pt'
dst_obj = f'{drive_weights}/yolo_objectness_best.pt'
shutil.copy2(src_obj, dst_obj)
print(f'Saved: {dst_obj}  ({os.path.getsize(dst_obj) / 1e6:.1f} MB)')

# Save full training results (logs, charts, curves)
drive_results = f'{DRIVE}/results'
shutil.copytree('/content/runs/yolo_14class/train', f'{drive_results}/yolo_14class', dirs_exist_ok=True)
shutil.copytree('/content/runs/yolo_objectness/train', f'{drive_results}/yolo_objectness', dirs_exist_ok=True)
print(f'\nTraining results saved to: {drive_results}/')

print(f'\n=== Your Google Drive SnapShelf folder now contains ===')
print(f'  SnapShelf/')
print(f'  ├── dataset_14class.zip       (uploaded by you)')
print(f'  ├── dataset_objectness.zip    (uploaded by you)')
print(f'  ├── weights/')
print(f'  │   ├── yolo_14class_best.pt  ← download this')
print(f'  │   └── yolo_objectness_best.pt ← download this')
print(f'  └── results/')
print(f'      ├── yolo_14class/         (training logs, charts)')
print(f'      └── yolo_objectness/      (training logs, charts)')

## Step 7: Quick Validation

Run a quick test to make sure the models work.

In [ ]:
# Test 14-class model on a random val image
import glob
val_images = glob.glob('/content/dataset/val/images/*')
test_img = val_images[0]
print(f'Testing on: {test_img}')

model_test = YOLO(dst_14)
results = model_test.predict(test_img, conf=0.25, verbose=False)
for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        name = model_test.names[cls_id]
        print(f'  Detected: {name} ({conf:.2f})')

print('\n14-class model OK!')

# Test objectness model
model_test2 = YOLO(dst_obj)
results2 = model_test2.predict(test_img, conf=0.25, verbose=False)
for r in results2:
    print(f'  Objectness detections: {len(r.boxes)}')

print('Objectness model OK!')
print('\n=== ALL DONE — download weights from Google Drive ===')